## MD 正文断行：检测与可选合并写回

**运行方式**：请在仓库根目录打开并运行本 Notebook（当前工作目录需能访问 `knowledgeBase`）。

**依赖**：仅 Python 标准库（`pathlib`、`re`、`glob` 等）。

**当前阶段**：第一个代码单元提供检测与合并用函数；`RUN_SCAN` / `RUN_FIX` 控制是否对 `md_to_chunk` 下两目录**批量检测** / **合并写回**（默认均为 `False`，避免 Run All 误跑）。

**跳过不检测的行**：空行、`#` 标题、`![](...)` 图片、`[^n]:` 脚注定义、**整行**为 `**…**` 的加粗行（如 `**【原文】**`）。

**正常行末**：去掉尾部若干 ` [^数字]` 脚注引用后，最后一个字符属于配置的标点集合；但若行尾为 `，`+`“`（逗号 + 左双引号）等后缀（`_LINE_END_INVALID_SUFFIXES`），仍视为异常断行。

**不报告的情况**：① 该行为全文件最后一个非空行；② 配置为「整行加粗豁免」且上一非空行是整行 `**…**`；③ `EXEMPT_PREV_HEADING` 为真且上一非空行为任意级 Markdown 标题（`#`…）（见下方代码常量）。

**取舍说明**：`EXEMPT_PREV_FULL_BOLD_MODE = "all"` 时，紧接 `**【原文】**` / `**【译文】**` 后的断行（如 `Ssc03_0002.md` 第 7 行）**不会**被报出。若要敏感检测，请改为 `"none"` 或 `"patterns"` 并配置 `FULL_BOLD_EXEMPT_EXACT`。

In [45]:
from __future__ import annotations

import glob
import re
from pathlib import Path
from typing import Any

# ---------------------------------------------------------------------------
# 行末标点（可按语料微调）
# ---------------------------------------------------------------------------
_LINE_END_PUNCT_BASE = (
    "。！？；：”…—·「」『』（）【】《》〈〉＂＇.?!;:\"'()[]”"
)
LINE_END_PUNCT = frozenset(_LINE_END_PUNCT_BASE + "\u201c\u201d\u2018\u2019")

# 即使字符在 LINE_END_PUNCT 中，仍视为「不是合法行尾」（可按语料微调；默认空）
# 例：古文常在中逗、顿号处折行时，可设为 frozenset("，、")
LINE_END_NOT_PUNCT = frozenset()

# 行尾为下列后缀时，即便最后一字在 LINE_END_PUNCT 内也视为异常（如：逗号后开左引号却被折行）
_LINE_END_INVALID_SUFFIXES = ("\u201c",)

# ---------------------------------------------------------------------------
# 豁免：上一非空行为整行加粗时，是否不报告当前正文行
#   "all"     — 任意整行 **…** 均豁免其下一正文行（会漏检 **【原文】** 后断行）
#   "none"    — 不使用该豁免
#   "patterns"— 仅当上一非空行 strip() 精确属于 FULL_BOLD_EXEMPT_EXACT 时豁免
# ---------------------------------------------------------------------------
EXEMPT_PREV_FULL_BOLD_MODE = "all"
FULL_BOLD_EXEMPT_EXACT = frozenset({"**【梁注】**", "**【注释】**"})

# 上一非空行为 Markdown 标题（#… 任意级）时，不报告当前正文行
EXEMPT_PREV_HEADING = True

_RE_TRAILING_FOOTNOTES = re.compile(r"(?:\s+\[\^\d+\])+\s*$")
_RE_FULL_BOLD = re.compile(r"^\*\*.+\*\*$")
_RE_FOOTNOTE_DEF = re.compile(r"^\[\^\d+\]:")


def read_lines(path: Path) -> list[str]:
    text = path.read_text(encoding="utf-8")
    return text.splitlines()


def is_full_bold_line(line: str) -> bool:
    s = line.strip()
    return bool(s and _RE_FULL_BOLD.match(s))


def is_skipped_line(line: str) -> bool:
    s = line.strip()
    if not s:
        return True
    if s.startswith("#"):
        return True
    if s.startswith("!["):
        return True
    if _RE_FOOTNOTE_DEF.match(s):
        return True
    if is_full_bold_line(line):
        return True
    return False


def strip_trailing_footnote_refs(line: str) -> str:
    s = line.strip()
    while True:
        new_s = _RE_TRAILING_FOOTNOTES.sub("", s)
        new_s = new_s.rstrip()
        if new_s == s:
            break
        s = new_s
    return s


def line_ends_normally(line: str) -> bool:
    core = strip_trailing_footnote_refs(line)
    if not core:
        return False
    if any(core.endswith(suf) for suf in _LINE_END_INVALID_SUFFIXES):
        return False
    last = core[-1]
    if last in LINE_END_NOT_PUNCT:
        return False
    return last in LINE_END_PUNCT


def prev_nonempty_line(lines: list[str], idx: int) -> str | None:
    j = idx - 1
    while j >= 0:
        if lines[j].strip():
            return lines[j]
        j -= 1
    return None


def last_nonempty_index(lines: list[str]) -> int | None:
    for i in range(len(lines) - 1, -1, -1):
        if lines[i].strip():
            return i
    return None


def _prev_full_bold_exempts_current(lines: list[str], idx: int) -> bool:
    prev = prev_nonempty_line(lines, idx)
    if prev is None:
        return False
    mode = EXEMPT_PREV_FULL_BOLD_MODE
    if mode == "none":
        return False
    if mode == "all":
        return is_full_bold_line(prev)
    if mode == "patterns":
        ps = prev.strip()
        return ps in FULL_BOLD_EXEMPT_EXACT
    raise ValueError(f"未知 EXEMPT_PREV_FULL_BOLD_MODE: {mode!r}")


def _is_heading_line(line: str) -> bool:
    s = line.lstrip()
    return bool(s.startswith("#"))


def _prev_heading_exempts_current(lines: list[str], idx: int) -> bool:
    if not EXEMPT_PREV_HEADING:
        return False
    prev = prev_nonempty_line(lines, idx)
    if prev is None:
        return False
    return _is_heading_line(prev)


def should_flag_body_line(
    lines: list[str], idx: int, last_nz: int | None = None
) -> bool:
    if last_nz is None:
        last_nz = last_nonempty_index(lines)
    line = lines[idx]
    if is_skipped_line(line):
        return False
    if line_ends_normally(line):
        return False
    if last_nz is not None and idx == last_nz:
        return False
    if _prev_full_bold_exempts_current(lines, idx):
        return False
    if _prev_heading_exempts_current(lines, idx):
        return False
    return True


def detect_bad_breaks(path: Path) -> list[dict[str, Any]]:
    lines = read_lines(path)
    last_nz = last_nonempty_index(lines)
    issues: list[dict[str, Any]] = []
    for i, line in enumerate(lines):
        if not should_flag_body_line(lines, i, last_nz):
            continue
        preview = line.strip()
        if len(preview) > 120:
            preview = preview[:117] + "..."
        issues.append(
            {
                "path": str(path.resolve()),
                "line_no": i + 1,
                "line_preview": preview,
                "reason": "no_terminal_punct_after_stripping_footnotes",
            }
        )
    return issues


def scan_paths(
    paths: list[str | Path],
    *,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    all_issues: list[dict[str, Any]] = []
    for p in paths:
        path = Path(p)
        if path.is_dir():
            md_files = sorted(path.glob("**/*.md"))
        else:
            md_files = [path]
        for f in md_files:
            if not f.is_file():
                continue
            issues = detect_bad_breaks(f)
            all_issues.extend(issues)
            if verbose and issues:
                print(f"\n{f} — {len(issues)} 处")
                for it in issues:
                    print(f"  L{it['line_no']}: {it['line_preview']}")
    if verbose:
        print(f"\n合计: {len(all_issues)} 条疑似错误分段")
    return all_issues


def scan_glob(pattern: str, *, verbose: bool = True) -> list[dict[str, Any]]:
    files = [Path(p) for p in sorted(glob.glob(pattern, recursive=True))]
    files = [p for p in files if p.is_file() and p.suffix.lower() == ".md"]
    return scan_paths(files, verbose=verbose)


# ---------------------------------------------------------------------------
# 待扫描目录：`knowledgeBase/md_to_chunk/` 下两份子目录（递归 *.md）
# ---------------------------------------------------------------------------
# 自测预期（若改为单文件调试）：
# - Ssc03_0002：mode="all" 时第 7、31 行**不报**（整行加粗豁免）；
#   改为 "none" 或 "patterns"（且不含 **【原文】**）后应能报出。
# - Ssc08_0006：正文多句号结尾，误报应较少。
# - Ssc07_0001：带 [^1] [^2] 行尾应判正常。
# 将 RUN_SCAN=True 再运行本 cell，可对 MD_CHUNK_DIRS 递归检测（默认 False，避免 Run All 刷屏）。
# ---------------------------------------------------------------------------
ROOT = Path(".")
MD_CHUNK_DIRS = [
    ROOT / "knowledgeBase/md_to_chunk/liangzhu_Markdown_2chunk",
    ROOT / "knowledgeBase/md_to_chunk/wangzhu_Markdown_Output",
]
RUN_SCAN = True

existing_dirs = [p for p in MD_CHUNK_DIRS if p.is_dir()]
if RUN_SCAN and existing_dirs:
    _ = scan_paths(existing_dirs, verbose=True)
elif RUN_SCAN:
    print("未找到待扫描目录，请将 ROOT 设为仓库根目录或修改 MD_CHUNK_DIRS。")
elif not existing_dirs:
    print("未找到待扫描目录；将 RUN_SCAN=True 后重新运行本 cell 可全库检测。")
else:
    print("已跳过扫描（RUN_SCAN=False）。改为 True 后重新运行本 cell。")



合计: 0 条疑似错误分段


## 合并错误分段并写回

1. **先运行**上一代码单元（加载函数）。需要全库检测报告时，在同一单元内将 `RUN_SCAN = True` 后再运行一次。
2. 本单元将 `RUN_FIX = True` 后**单独运行**，按与检测相同的规则向下合并正文行并**直接覆盖**对应 `.md`。
3. 建议写回前用 Git 提交或备份；合并时中间的空行会被丢弃，遇标题 / 整行加粗 / 图片 / 脚注定义则停止向下拼接。


In [43]:
# 依赖：先运行上一单元（检测逻辑与常量）。
# 将 `should_flag_body_line` 为真的正文行向下与后续正文行拼接（中间空行丢弃），
# 直到 `line_ends_normally` 为真，或遇到标题/整行加粗/图片/脚注定义/文末。
# 多遍扫描直到本遍无合并，**直接覆盖写回**原 .md。

from pathlib import Path


def _merge_one_pass(lines: list[str]) -> tuple[list[str], int]:
    last_nz = last_nonempty_index(lines)
    out: list[str] = []
    i = 0
    merge_count = 0
    while i < len(lines):
        line = lines[i]
        if is_skipped_line(line):
            out.append(line)
            i += 1
            continue
        if not should_flag_body_line(lines, i, last_nz):
            out.append(line)
            i += 1
            continue
        acc = line.rstrip()
        j = i + 1
        while j < len(lines):
            if line_ends_normally(acc):
                break
            nxt = lines[j]
            if not nxt.strip():
                j += 1
                continue
            if is_skipped_line(nxt):
                break
            acc = acc + nxt.strip()
            merge_count += 1
            j += 1
        out.append(acc)
        i = j
    return out, merge_count


def fix_paragraph_breaks_in_file(path: Path) -> int:
    raw = path.read_text(encoding="utf-8")
    ends_nl = raw == "" or raw.endswith("\n")
    lines = raw.splitlines()
    total_merges = 0
    while True:
        lines, n = _merge_one_pass(lines)
        total_merges += n
        if n == 0:
            break
    text = "\n".join(lines)
    if ends_nl:
        text += "\n"
    path.write_text(text, encoding="utf-8")
    return total_merges


def fix_paragraph_breaks_paths(
    paths: list[str | Path],
    *,
    verbose: bool = True,
) -> dict[str, int]:
    stats: dict[str, int] = {}
    for p in paths:
        path = Path(p)
        md_files = sorted(path.glob("**/*.md")) if path.is_dir() else [path]
        for f in md_files:
            if not f.is_file():
                continue
            n = fix_paragraph_breaks_in_file(f)
            stats[str(f.resolve())] = n
            if verbose and n:
                print(f"{f}：合并次数 {n}")
    if verbose:
        touched = sum(1 for v in stats.values() if v)
        merges = sum(stats.values())
        print(f"完成：共 {touched} 个文件有修改，累计合并 {merges} 次")
    return stats


# 与检测单元相同的目录。将 RUN_FIX 改为 True 再运行本 cell，才会**覆盖写回** md（避免「Run All」误改文件）。
ROOT = Path(".")
MD_CHUNK_DIRS_FIX = [
    ROOT / "knowledgeBase/md_to_chunk/liangzhu_Markdown_2chunk",
    ROOT / "knowledgeBase/md_to_chunk/wangzhu_Markdown_Output",
]
RUN_FIX = True

_dirs = [p for p in MD_CHUNK_DIRS_FIX if p.is_dir()]
if RUN_FIX and _dirs:
    _ = fix_paragraph_breaks_paths(_dirs, verbose=True)
elif RUN_FIX:
    print("未找到目录，请设置 ROOT 或 MD_CHUNK_DIRS_FIX。")
elif not _dirs:
    print("未找到目录；若仅调试函数，可忽略。将 RUN_FIX=True 后写回。")
else:
    print("已跳过写回（RUN_FIX=False）。改为 True 后重新运行本 cell。")


knowledgeBase\md_to_chunk\liangzhu_Markdown_2chunk\Ssc05_0001.md：合并次数 3
knowledgeBase\md_to_chunk\liangzhu_Markdown_2chunk\Ssc05_0002.md：合并次数 4
完成：共 2 个文件有修改，累计合并 7 次


### 新cell：判断每个md文件下的同一等级标题下的原文段落数和译文段落数是否相等

1. 从上至下遍历，匹配**【原文】**内容，然后计算其下方的正文段落的数量，独立的图片行不计数，直到遇到下一个类型行停止。
2. 计入数量后，匹配**【译文】**内容，同样逻辑，计算其下方正文段落的数量，独立的图片行不计数，直到遇到下一个类型行停止。
3. 原文 译文 为一对，返回每个md文件包含几个原文译文对，每个原文译文对的数量是否相等，该原文译文对的最近标题是什么，最近标题通过查找距离该原文译文对最近的一个标题行确认


In [56]:
# 依赖：先运行「检测」单元（read_lines、is_full_bold_line、_RE_FOOTNOTE_DEF）。
# 统计每个 **【原文】** / **【译文】** 块内「正文段落」数（空行分段；独立 ![](...) 行不计数、不分段）；
# 第 i 个【原文】与第 i 个【译文】配对；最近标题为该行【原文】之前最后一个 # 标题行。

from __future__ import annotations

import re
from pathlib import Path
from typing import Any

_RE_ORIG = re.compile(r"【原文】")
_RE_TRANS = re.compile(r"【译文】")


def _is_marker_original(line: str) -> bool:
    s = line.strip()
    return bool(s and is_full_bold_line(line) and _RE_ORIG.search(s))


def _is_marker_translation(line: str) -> bool:
    s = line.strip()
    return bool(s and is_full_bold_line(line) and _RE_TRANS.search(s))


def _is_section_boundary_line(line: str) -> bool:
    """遇到则结束当前【原文】/【译文】正文区间（不计入本块段落）。"""
    s = line.strip()
    if not s:
        return False
    if s.startswith("#"):
        return True
    if is_full_bold_line(line):
        return True
    if _RE_FOOTNOTE_DEF.match(s):
        return True
    return False


def _is_standalone_image_line(line: str) -> bool:
    return bool(line.strip().startswith("!["))


def count_body_paragraphs_until_boundary(lines: list[str], start_after: int) -> tuple[int, int]:
    """
    从 start_after 的下一行开始扫描，直到遇到类型行（不含空行、不含独立图片）。
    返回 (段落数, 结束行索引：第一个边界行的下标，若扫到文件尾则为 len(lines))。
    段落：非空、非独立图片的正文行，以「一个或多个空行」分隔为不同段；段内多行合并为一段。
    """
    i = start_after + 1
    para_count = 0
    in_para = False
    while i < len(lines):
        line = lines[i]
        if _is_section_boundary_line(line):
            break
        if not line.strip():
            if in_para:
                para_count += 1
                in_para = False
            i += 1
            continue
        if _is_standalone_image_line(line):
            i += 1
            continue
        in_para = True
        i += 1
    if in_para:
        para_count += 1
    return para_count, i


def _nearest_heading_before(lines: list[str], idx: int) -> str | None:
    j = idx - 1
    while j >= 0:
        s = lines[j].lstrip()
        if s.startswith("#"):
            return lines[j].strip()
        j -= 1
    return None


def analyze_original_translation_pairs(path: Path) -> dict[str, Any]:
    lines = read_lines(path)
    orig_indices: list[int] = []
    trans_indices: list[int] = []
    for i, line in enumerate(lines):
        if _is_marker_original(line):
            orig_indices.append(i)
        elif _is_marker_translation(line):
            trans_indices.append(i)

    n_o, n_t = len(orig_indices), len(trans_indices)
    n_pair = min(n_o, n_t)
    pairs: list[dict[str, Any]] = []
    for k in range(n_pair):
        oi, ti = orig_indices[k], trans_indices[k]
        n_orig, _ = count_body_paragraphs_until_boundary(lines, oi)
        n_trans, _ = count_body_paragraphs_until_boundary(lines, ti)
        pairs.append(
            {
                "pair_index": k + 1,
                "original_line": oi + 1,
                "translation_line": ti + 1,
                "original_paragraphs": n_orig,
                "translation_paragraphs": n_trans,
                "counts_equal": n_orig == n_trans,
                "nearest_heading": _nearest_heading_before(lines, oi),
            }
        )

    return {
        "path": str(path.resolve()),
        "marker_original_count": n_o,
        "marker_translation_count": n_t,
        "paired_count": n_pair,
        "unpaired_original": n_o - n_pair,
        "unpaired_translation": n_t - n_pair,
        "pairs": pairs,
        "all_pairs_equal": all(p["counts_equal"] for p in pairs) if pairs else True,
    }


def scan_original_translation_pairs(
    paths: list[str | Path],
    *,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    for p in paths:
        path = Path(p)
        md_files = sorted(path.glob("**/*.md")) if path.is_dir() else [path]
        for f in md_files:
            if not f.is_file():
                continue
            results.append(analyze_original_translation_pairs(f))
    if verbose:
        def _needs_attention(r: dict[str, Any]) -> bool:
            return (
                (not r["all_pairs_equal"])
                or r["unpaired_original"]
                or r["unpaired_translation"]
            )

        mismatch_files = [r for r in results if _needs_attention(r)]
        ok_count = len(results) - len(mismatch_files)
        print(
            f"共扫描 {len(results)} 个 .md；需列出 {len(mismatch_files)} 个；"
            f"标记对齐且各对段落数均相等 {ok_count} 个（不打印）。\n"
        )
        for r in mismatch_files:
            rel = Path(r["path"]).name
            print(
                f" [!] {rel} — 原文标记×{r['marker_original_count']} "
                f"译文标记×{r['marker_translation_count']} 已配对{r['paired_count']}对"
            )
            if r["unpaired_original"] or r["unpaired_translation"]:
                print(
                    f"      未配对: 多余原文标记 {r['unpaired_original']}，多余译文标记 {r['unpaired_translation']}"
                )
            for p in r["pairs"]:
                eq = "相等" if p["counts_equal"] else "不等"
                h = p["nearest_heading"] or "(文件首无标题)"
                print(
                    f"      第{p['pair_index']}对 L{p['original_line']}/L{p['translation_line']}: "
                    f"原文{p['original_paragraphs']}段 / 译文{p['translation_paragraphs']}段 — {eq}；最近标题: {h}"
                )
            print()
    return results


ROOT = Path(".")
MD_CHUNK_DIRS_PAIR = [
    ROOT / "knowledgeBase/md_to_chunk/liangzhu_Markdown_2chunk",
    ROOT / "knowledgeBase/md_to_chunk/wangzhu_Markdown_Output",
]
RUN_PAIR_CHECK = True

_dirs_p = [p for p in MD_CHUNK_DIRS_PAIR if p.is_dir()]
if RUN_PAIR_CHECK and _dirs_p:
    pair_results = scan_original_translation_pairs(_dirs_p, verbose=True)
elif RUN_PAIR_CHECK:
    print("未找到目录，请将 ROOT 设为仓库根目录或修改 MD_CHUNK_DIRS_PAIR。")
else:
    print("已跳过（RUN_PAIR_CHECK=False）。改为 True 后运行本 cell。")

共扫描 74 个 .md；需列出 32 个；标记对齐且各对段落数均相等 42 个（不打印）。

 [!] Ssc08_0001.md — 原文标记×14 译文标记×14 已配对14对
      第1对 L6/L126: 原文12段 / 译文12段 — 相等；最近标题: ### 板门双扇板门、独扇板门
      第2对 L158/L240: 原文23段 / 译文22段 — 不等；最近标题: ### 乌头门
      第3对 L289/L343: 原文11段 / 译文11段 — 相等；最近标题: ### 软门牙头护缝软门、合板软门
      第4对 L370/L412: 原文7段 / 译文7段 — 相等；最近标题: ### 破子棂窗
      第5对 L431/L457: 原文5段 / 译文5段 — 相等；最近标题: ### 睒电窗
      第6对 L471/L497: 原文8段 / 译文8段 — 相等；最近标题: ### 板棂窗
      第7对 L518/L554: 原文10段 / 译文10段 — 相等；最近标题: ### 截间板帐
      第8对 L581/L633: 原文16段 / 译文17段 — 不等；最近标题: ### 照壁屏风骨
      第9对 L672/L690: 原文5段 / 译文5段 — 相等；最近标题: ### 隔截横钤立旌
      第10对 L707/L749: 原文8段 / 译文8段 — 相等；最近标题: ### 露篱
      第11对 L770/L792: 原文7段 / 译文7段 — 相等；最近标题: ### 板引檐
      第12对 L811/L831: 原文7段 / 译文7段 — 相等；最近标题: ### 水槽
      第13对 L849/L921: 原文21段 / 译文21段 — 相等；最近标题: ### 井屋子
      第14对 L968/L992: 原文6段 / 译文7段 — 不等；最近标题: ### 地棚

 [!] Ssc08_0002.md — 原文标记×13 译文标记×13 已配对13对
      第1对 L8/L118: 原文21段 / 译文20段 — 不等；最近标题: ### 格子门
      第2对 L163/L211: 原文16段 / 译文16段 — 相等；最近标题: 